# 05 Models — CatBoost — Men's

CatBoost as an additional gradient boosting variant. CatBoost handles missing values natively and uses ordered boosting to reduce overfitting.

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/catboost/mens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from catboost import CatBoostClassifier, Pool
except:
    !pip install catboost
    from catboost import CatBoostClassifier, Pool

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

import catboost
print(f"CatBoost version: {catboost.__version__}")

CatBoost version: 1.2.10


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/catboost"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (5170, 82)
Stage 1 grid: (261013, 80)
Stage 2 grid: (66430, 79)
Features: 27


## 2. CatBoost Hyperparameters

In [6]:
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'depth': 4,
    'learning_rate': 0.05,
    'l2_leaf_reg': 3.0,
    'iterations': 500,
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 0,
}

print("CatBoost parameters:")
for k, v in cb_params.items():
    print(f"  {k}: {v}")

CatBoost parameters:
  loss_function: Logloss
  eval_metric: Logloss
  depth: 4
  learning_rate: 0.05
  l2_leaf_reg: 3.0
  iterations: 500
  early_stopping_rounds: 50
  random_seed: 42
  verbose: 0


## 3. LOGO Cross-Validation

In [7]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    model = CatBoostClassifier(**cb_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        verbose=False
    )
    
    preds = model.predict_proba(X_val)[:, 1]
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration_
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration_}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1985: Brier=0.1826, Games=63, BestRound=234
  Fold 1986: Brier=0.2030, Games=63, BestRound=52
  Fold 1987: Brier=0.1759, Games=63, BestRound=330
  Fold 1988: Brier=0.1950, Games=63, BestRound=94
  Fold 1989: Brier=0.1720, Games=63, BestRound=324
  Fold 1990: Brier=0.1896, Games=63, BestRound=329
  Fold 1991: Brier=0.1820, Games=63, BestRound=318
  Fold 1992: Brier=0.1678, Games=63, BestRound=255
  Fold 1993: Brier=0.1605, Games=63, BestRound=264
  Fold 1994: Brier=0.1636, Games=63, BestRound=433
  Fold 1995: Brier=0.1695, Games=63, BestRound=219
  Fold 1996: Brier=0.1637, Games=63, BestRound=111
  Fold 1997: Brier=0.1936, Games=63, BestRound=64
  Fold 1998: Brier=0.1927, Games=63, BestRound=108
  Fold 1999: Brier=0.1960, Games=63, BestRound=298
  Fold 2000: Brier=0.1828, Games=63, BestRound=162
  Fold 2001: Brier=0.1992, Games=64, BestRound=57
  Fold 2002: Brier=0.1912, Games=64, BestRound=210
  Fold 2003: Brier=0.1785, Games=64, BestRound=328
  Fold 2004: Brier=0.1750, Games=64

In [8]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")


Overall OOF Brier Score: 0.1837
Stage 1 (2022-2025) Brier Score: 0.1908
Mean Brier: 0.1835 +/- 0.0188


## 4. Train Final Model and Calibrate

In [9]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds}")

final_params = cb_params.copy()
final_params['iterations'] = final_rounds
final_params.pop('early_stopping_rounds', None)

final_model = CatBoostClassifier(**final_params)
final_model.fit(train[feature_cols], train['Label'], verbose=False)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

Final model rounds: 156
OOF Brier (raw): 0.1837
OOF Brier (calibrated): 0.1802


## 5. Generate Predictions

In [10]:
stage1['Pred_catboost'] = final_model.predict_proba(stage1[feature_cols])[:, 1]
stage1['Pred_catboost_calibrated'] = calibrator.predict(stage1['Pred_catboost'].values).clip(0.02, 0.98)

stage2['Pred_catboost'] = final_model.predict_proba(stage2[feature_cols])[:, 1]
stage2['Pred_catboost_calibrated'] = calibrator.predict(stage2['Pred_catboost'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_catboost_calibrated'].min():.3f}, {stage1['Pred_catboost_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_catboost_calibrated'].min():.3f}, {stage2['Pred_catboost_calibrated'].max():.3f}]")

stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_catboost_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.173, 0.980]
Stage 1 Brier (calibrated): 0.1790


## 6. Save Outputs

In [11]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_catboost', 'Pred_catboost_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_catboost', 'Pred_catboost_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/catboost/mens/cv_results.parquet ((40, 4))


## 7. Summary

In [12]:
print("=" * 60)
print("CATBOOST MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")

CATBOOST MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.1837
OOF Brier Score (calibrated): 0.1802
CV Mean Brier: 0.1835 +/- 0.0188
Final model rounds: 156
